## 1) Install and import packages

In Colab, upload these three files first:
- `train.csv`
- `test.csv`
- `cleaner.py`

In [ ]:
!pip -q install xgboost

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, GridSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor

from xgboost import XGBRegressor

## 2) Upload files in Colab

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
from cleaner import HouseCleaner

## 3) Load data

In [ ]:
df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)
df_train.head()

Train shape: (1241, 81)
Test shape: (219, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,985,90,RL,75.0,10125,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2009,COD,Normal,125969
1,778,20,RL,100.0,13350,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,6,2006,WD,Normal,142555
2,708,120,RL,48.0,6240,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2009,WD,Normal,254214
3,599,20,RL,80.0,12984,Pave,NaN,Reg,Bnk,AllPub,...,0,NaN,NaN,NaN,0,3,2006,WD,Normal,217309
4,875,50,RM,52.0,5720,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2009,WD,Abnorml,66406


## 4) Preprocess with your custom cleaner

In [ ]:
cleaner = HouseCleaner()

X_raw = df_train.drop("SalePrice", axis=1)
y = df_train["SalePrice"]

X_clean = cleaner.fit_transform(X_raw)
X_test_clean = cleaner.transform(df_test)

feature_names = [f"f{i}" for i in range(X_clean.shape[1])]
X = pd.DataFrame(X_clean, columns=feature_names, index=df_train.index)
X_test_final = pd.DataFrame(X_test_clean, columns=feature_names, index=df_test.index)

print("Cleaned train shape:", X.shape)
print("Cleaned test shape:", X_test_final.shape)
print("Numeric matrix:", np.issubdtype(X.values.dtype, np.number))
print("Missing values:", np.isnan(X.values).sum())

Cleaned train shape: (1241, 195)
Cleaned test shape: (219, 195)
Numeric matrix: True
Missing values: 0


## 5) Train / validation split

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=42
)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

## 6) Utility functions

In [ ]:
def print_metrics(y_true, y_pred, name):
    print(f"\n{name}")
    print("-" * len(name))
    print("MAE :", mean_absolute_error(y_true, y_pred))
    print("MSE :", mean_squared_error(y_true, y_pred))
    print("R2  :", r2_score(y_true, y_pred))


def evaluate_model(model, X_train, y_train, X_valid, y_valid, name):
    model.fit(X_train, y_train)
    valid_pred = model.predict(X_valid)
    train_pred = model.predict(X_train)

    print_metrics(y_valid, valid_pred, f"{name} - Validation")
    print_metrics(y_train, train_pred, f"{name} - Train")

    return {
        "model": name,
        "valid_mae": mean_absolute_error(y_valid, valid_pred),
        "train_mae": mean_absolute_error(y_train, train_pred),
        "valid_r2": r2_score(y_valid, valid_pred),
        "train_r2": r2_score(y_train, train_pred)
    }


def summarize_search(search, name):
    results = pd.DataFrame(search.cv_results_)
    cols = [
        "rank_test_score",
        "mean_test_score",
        "std_test_score",
        "mean_fit_time",
        "params"
    ]
    print(f"\n{name} best params:")
    print(search.best_params_)
    print(f"{name} best CV MAE:", -search.best_score_)
    return results[cols].sort_values("rank_test_score").head(10)

## 7) Baseline models

In [ ]:
baseline_rf = RandomForestRegressor(random_state=42, n_jobs=-1)
baseline_gb = GradientBoostingRegressor(random_state=42)

baseline_results = []
baseline_results.append(
    evaluate_model(baseline_rf, X_train, y_train, X_valid, y_valid, "Baseline Random Forest")
)
baseline_results.append(
    evaluate_model(baseline_gb, X_train, y_train, X_valid, y_valid, "Baseline Gradient Boosting")
)

pd.DataFrame(baseline_results).sort_values("valid_mae")


Baseline Random Forest - Validation
-----------------------------------
MAE : 17993.506265060238
MSE : 928693145.0258569
R2  : 0.8698936473703052

Baseline Random Forest - Train
------------------------------
MAE : 6637.055776209677
MSE : 126628615.64532669
R2  : 0.9777261700162996

Baseline Gradient Boosting - Validation
---------------------------------------
MAE : 17045.180537592707
MSE : 831883410.6026043
R2  : 0.8834563203719542

Baseline Gradient Boosting - Train
----------------------------------
MAE : 9879.16180766493
MSE : 175591805.80353966
R2  : 0.9691136003574929


,model,valid_mae,train_mae,valid_r2,train_r2
1,Baseline Gradient Boosting,17045.180538,9879.161808,0.883456,0.969114
0,Baseline Random Forest,17993.506265,6637.055776,0.869894,0.977726


In [ ]:
cv_scores_rf_baseline = -cross_val_score(
    baseline_rf,
    X_train,
    y_train,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

print("Baseline RF CV MAE:", cv_scores_rf_baseline.mean())
print("Std:", cv_scores_rf_baseline.std())

Baseline RF CV MAE: 18670.353191716153
Std: 2342.0325479283106


## 8) Random Forest — GridSearchCV

This grid is much cleaner than the earlier notebook and stays realistic for Colab.

In [ ]:
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

# rf_param_grid = {
#     "n_estimators": [300, 600, 900],
#     "max_depth": [None, 10, 20],
#     "min_samples_split": [2, 5, 10],
#     "min_samples_leaf": [1, 2, 4],
#     "max_features": ["sqrt", 0.7],
#     "bootstrap": [True]
# }

rf_param_grid = {
    "n_estimators": [200, 400, 600],
    "max_depth": [None, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 0.7],
    "bootstrap": [True]
}
grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=rf_param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    refit=True
)

grid_rf.fit(X_train, y_train)
rf_top = summarize_search(grid_rf, "Random Forest")
rf_top

Fitting 5 folds for each of 48 candidates, totalling 240 fits

Random Forest best params:
{'bootstrap': True, 'max_depth': 20, 'max_features': 0.7, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 400}
Random Forest best CV MAE: 17971.040522710857


,rank_test_score,mean_test_score,std_test_score,mean_fit_time,params
37,1,-17971.040523,1699.818848,12.241097,"{'bootstrap': True, 'max_depth': 20, 'max_feat..."
13,2,-17973.941519,1740.960128,12.316209,"{'bootstrap': True, 'max_depth': None, 'max_fe..."
14,3,-17974.510850,1761.131000,17.741653,"{'bootstrap': True, 'max_depth': None, 'max_fe..."
38,4,-17981.983715,1719.417454,17.629117,"{'bootstrap': True, 'max_depth': 20, 'max_feat..."
36,5,-17984.792532,1745.091091,6.165391,"{'bootstrap': True, 'max_depth': 20, 'max_feat..."
44,6,-17997.356268,1815.501773,13.546325,"{'bootstrap': True, 'max_depth': 20, 'max_feat..."
43,7,-18000.975819,1793.437125,8.838317,"{'bootstrap': True, 'max_depth': 20, 'max_feat..."
20,8,-18004.077999,1814.862516,13.623034,"{'bootstrap': True, 'max_depth': None, 'max_fe..."
19,9,-18008.053016,1790.822249,8.933161,"{'bootstrap': True, 'max_depth': None, 'max_fe..."
21,10,-18012.213298,1838.881848,4.361628,"{'bootstrap': True, 'max_depth': None, 'max_fe..."


In [ ]:
best_rf = grid_rf.best_estimator_
rf_result = evaluate_model(best_rf, X_train, y_train, X_valid, y_valid, "Tuned Random Forest")


Tuned Random Forest - Validation
--------------------------------
MAE : 18194.89391265235
MSE : 944409920.8059541
R2  : 0.8676917872803501

Tuned Random Forest - Train
---------------------------
MAE : 6463.611680236315
MSE : 120039899.65466829
R2  : 0.9788851176920592


## 9) Gradient Boosting — GridSearchCV

Your old notebook mixed multiple randomized runs. This version keeps one stable search block.

In [ ]:
gb = GradientBoostingRegressor(
    random_state=42,
    validation_fraction=0.1,
    n_iter_no_change=10,
    tol=1e-4
)

gb_param_grid = {
    "n_estimators": [200, 400, 600],
    "learning_rate": [0.02, 0.03, 0.05],
    "max_depth": [2, 3],
    "min_samples_split": [10, 20],
    "min_samples_leaf": [3, 8],
    "subsample": [0.70, 0.85],
    "max_features": ["sqrt", 0.8],
    "loss": ["huber"]
}

grid_gb = GridSearchCV(
    estimator=gb,
    param_grid=gb_param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    refit=True
)

grid_gb.fit(X_train, y_train)
gb_top = summarize_search(grid_gb, "Gradient Boosting")
gb_top

Fitting 5 folds for each of 288 candidates, totalling 1440 fits

Gradient Boosting best params:
{'learning_rate': 0.05, 'loss': 'huber', 'max_depth': 3, 'max_features': 'sqrt', 'min_samples_leaf': 8, 'min_samples_split': 10, 'n_estimators': 400, 'subsample': 0.85}
Gradient Boosting best CV MAE: 15849.635965946649


,rank_test_score,mean_test_score,std_test_score,mean_fit_time,params
257,1,-15849.635966,2086.275468,1.955440,"{'learning_rate': 0.05, 'loss': 'huber', 'max_..."
255,1,-15849.635966,2086.275468,2.674312,"{'learning_rate': 0.05, 'loss': 'huber', 'max_..."
263,3,-15977.602312,1520.582269,1.801220,"{'learning_rate': 0.05, 'loss': 'huber', 'max_..."
167,4,-15988.256802,1863.646332,3.429772,"{'learning_rate': 0.03, 'loss': 'huber', 'max_..."
64,5,-16005.094561,1822.635577,3.829688,"{'learning_rate': 0.02, 'loss': 'huber', 'max_..."
261,6,-16013.265459,1587.584004,1.933165,"{'learning_rate': 0.05, 'loss': 'huber', 'max_..."
148,7,-16015.749665,1819.355612,2.350699,"{'learning_rate': 0.03, 'loss': 'huber', 'max_..."
160,8,-16058.878572,1947.972499,2.986842,"{'learning_rate': 0.03, 'loss': 'huber', 'max_..."
146,9,-16090.326116,1902.983305,2.293491,"{'learning_rate': 0.03, 'loss': 'huber', 'max_..."
158,10,-16128.565370,1898.028203,2.510487,"{'learning_rate': 0.03, 'loss': 'huber', 'max_..."


In [ ]:
best_gb = grid_gb.best_estimator_
gb_result = evaluate_model(best_gb, X_train, y_train, X_valid, y_valid, "Tuned Gradient Boosting")


Tuned Gradient Boosting - Validation
------------------------------------
MAE : 17367.819805598607
MSE : 856106328.420736
R2  : 0.8800627823618528

Tuned Gradient Boosting - Train
-------------------------------
MAE : 12217.863727569287
MSE : 538900698.6581451
R2  : 0.9052079778426301


## 10) AdaBoost — GridSearchCV

We tune both the boosting stage and the weak learner.

In [ ]:
base_tree = DecisionTreeRegressor(random_state=42)

ab = AdaBoostRegressor(
    estimator=base_tree,
    random_state=42
)

ab_param_grid = {
    "n_estimators": [100, 300, 600],
    "learning_rate": [0.01, 0.03, 0.1],
    "loss": ["linear", "square"],
    "estimator__max_depth": [2, 3, 4],
    "estimator__min_samples_split": [2, 10],
    "estimator__min_samples_leaf": [1, 4]
}

grid_ab = GridSearchCV(
    estimator=ab,
    param_grid=ab_param_grid,
    scoring="neg_mean_absolute_error",
    cv=3,
    n_jobs=-1,
    verbose=1,
    refit=True
)

grid_ab.fit(X_train, y_train)
ab_top = summarize_search(grid_ab, "AdaBoost")
ab_top

Fitting 3 folds for each of 216 candidates, totalling 648 fits

AdaBoost best params:
{'estimator__max_depth': 4, 'estimator__min_samples_leaf': 4, 'estimator__min_samples_split': 10, 'learning_rate': 0.1, 'loss': 'linear', 'n_estimators': 600}
AdaBoost best CV MAE: 21212.567920468868


,rank_test_score,mean_test_score,std_test_score,mean_fit_time,params
212,1,-21212.567920,1170.352345,7.698665,"{'estimator__max_depth': 4, 'estimator__min_sa..."
158,2,-21222.566253,1190.971149,8.394141,"{'estimator__max_depth': 4, 'estimator__min_sa..."
176,3,-21277.723666,1279.766037,8.221676,"{'estimator__max_depth': 4, 'estimator__min_sa..."
194,4,-21286.423195,1257.655522,8.990266,"{'estimator__max_depth': 4, 'estimator__min_sa..."
157,5,-21309.151168,1220.397983,4.072573,"{'estimator__max_depth': 4, 'estimator__min_sa..."
193,6,-21377.566953,1278.312923,5.189743,"{'estimator__max_depth': 4, 'estimator__min_sa..."
214,7,-21398.241303,1291.317559,3.665626,"{'estimator__max_depth': 4, 'estimator__min_sa..."
211,8,-21409.677706,1201.372405,4.689810,"{'estimator__max_depth': 4, 'estimator__min_sa..."
175,9,-21414.491407,1399.864786,3.821466,"{'estimator__max_depth': 4, 'estimator__min_sa..."
179,10,-21431.708110,1283.494373,7.614453,"{'estimator__max_depth': 4, 'estimator__min_sa..."


In [ ]:
best_ab = grid_ab.best_estimator_
ab_result = evaluate_model(best_ab, X_train, y_train, X_valid, y_valid, "Tuned AdaBoost")


Tuned AdaBoost - Validation
---------------------------
MAE : 21479.545840213636
MSE : 1096055177.740038
R2  : 0.8464468676005164

Tuned AdaBoost - Train
----------------------
MAE : 16856.59155353196
MSE : 430859157.91828376
R2  : 0.9242123624152062


In [ ]:
xgb = XGBRegressor(
    objective="reg:squarederror",
    eval_metric="mae",
    tree_method="hist",
    random_state=42,
    n_jobs=2
)

xgb_param_grid = {
    "n_estimators": [300, 700, 1100],
    "learning_rate": [0.02, 0.05, 0.08],
    "max_depth": [2, 3, 4],
    "min_child_weight": [2, 5, 10],
    "subsample": [0.70, 0.85],
    "colsample_bytree": [0.70, 0.85],
    "reg_alpha": [0.0, 0.1],
    "reg_lambda": [1.0, 5.0]
}

grid_xgb = GridSearchCV(
    estimator=xgb,
    param_grid=xgb_param_grid,
    scoring="neg_mean_absolute_error",
    cv=3,
    n_jobs=2,
    verbose=1,
    refit=True
)

grid_xgb.fit(X_train, y_train)
xgb_top = summarize_search(grid_xgb, "XGBoost")
xgb_top

Fitting 3 folds for each of 1296 candidates, totalling 3888 fits

XGBoost best params:
{'colsample_bytree': 0.85, 'learning_rate': 0.02, 'max_depth': 4, 'min_child_weight': 2, 'n_estimators': 1100, 'reg_alpha': 0.1, 'reg_lambda': 5.0, 'subsample': 0.85}
XGBoost best CV MAE: 16775.5888671875


,rank_test_score,mean_test_score,std_test_score,mean_fit_time,params
815,1,-16775.588867,884.194345,3.924287,"{'colsample_bytree': 0.85, 'learning_rate': 0...."
811,2,-16775.634440,884.158109,3.694768,"{'colsample_bytree': 0.85, 'learning_rate': 0...."
812,3,-16818.934570,1020.618559,3.365886,"{'colsample_bytree': 0.85, 'learning_rate': 0...."
808,4,-16818.934896,1020.617123,4.295570,"{'colsample_bytree': 0.85, 'learning_rate': 0...."
814,5,-16820.419271,802.666487,3.137779,"{'colsample_bytree': 0.85, 'learning_rate': 0...."
810,6,-16820.421875,802.667825,3.760312,"{'colsample_bytree': 0.85, 'learning_rate': 0...."
164,7,-16822.636393,959.641451,3.522247,"{'colsample_bytree': 0.7, 'learning_rate': 0.0..."
160,8,-16823.405599,960.714802,3.148647,"{'colsample_bytree': 0.7, 'learning_rate': 0.0..."
303,9,-16837.991211,830.054298,1.420744,"{'colsample_bytree': 0.7, 'learning_rate': 0.0..."
299,9,-16837.991211,830.055837,1.415358,"{'colsample_bytree': 0.7, 'learning_rate': 0.0..."


In [ ]:
best_xgb = grid_xgb.best_estimator_
xgb_result = evaluate_model(best_xgb, X_train, y_train, X_valid, y_valid, "Tuned XGBoost")


Tuned XGBoost - Validation
--------------------------
MAE : 16363.5263671875
MSE : 796241856.0
R2  : 0.8884495496749878

Tuned XGBoost - Train
---------------------
MAE : 5479.9150390625
MSE : 57943736.0
R2  : 0.9898077845573425


## 12) Compare tuned models

In [ ]:
results_df = pd.DataFrame([rf_result, gb_result, ab_result, xgb_result])
results_df = results_df.sort_values("valid_mae").reset_index(drop=True)
results_df

,model,valid_mae,train_mae,valid_r2,train_r2
0,Tuned XGBoost,16363.526367,5479.915039,0.888450,0.989808
1,Tuned Gradient Boosting,17367.819806,12217.863728,0.880063,0.905208
2,Tuned Random Forest,18194.893913,6463.611680,0.867692,0.978885
3,Tuned AdaBoost,21479.545840,16856.591554,0.846447,0.924212


## 13) Refit the best model on the full cleaned training set

In [ ]:
best_model_name = results_df.loc[0, "model"]
print("Best model:", best_model_name)

model_lookup = {
    "Tuned Random Forest": best_rf,
    "Tuned Gradient Boosting": best_gb,
    "Tuned AdaBoost": best_ab,
    "Tuned XGBoost": best_xgb
}

final_model = model_lookup[best_model_name]
final_model.fit(X, y)

Best model: Tuned XGBoost


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.85, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric='mae', feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.02, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
             max_leaves=None, min_child_weight=2, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1100,
             n_jobs=2, num_parallel_tree=None, ...)

## 14) Create Kaggle submission

In [ ]:
submission = pd.DataFrame({
    "Id": df_test["Id"],
    "SalePrice": final_model.predict(X_test_final)
})

submission.to_csv("submission_gridsearch.csv", index=False)
submission.head()

,Id,SalePrice
0,893,141689.375000
1,1106,321888.468750
2,414,107010.148438
3,523,158839.156250
4,1037,311777.343750


In [ ]:
submission_new = pd.DataFrame({
    "Id": df_test["Id"],
    "SalePrice": best_gb.predict(X_test_final)
})

submission_new.to_csv("submission_gridsearch_gb.csv", index=False)

In [ ]:
submission_new_ab = pd.DataFrame({
    "Id": df_test["Id"],
    "SalePrice": best_ab.predict(X_test_final)
})

submission_new_ab.to_csv("submission_gridsearch_ab.csv", index=False)

## 15) Optional: save best parameter sets

In [ ]:
best_params_summary = {
    "RandomForest": grid_rf.best_params_,
    "GradientBoosting": grid_gb.best_params_,
    "AdaBoost": grid_ab.best_params_,
    "XGBoost": grid_xgb.best_params_
}

best_params_summary

{'RandomForest': {'bootstrap': True,
  'max_depth': 20,
  'max_features': 0.7,
  'min_samples_leaf': 1,
  'min_samples_split': 2,
  'n_estimators': 400},
 'GradientBoosting': {'learning_rate': 0.05,
  'loss': 'huber',
  'max_depth': 3,
  'max_features': 'sqrt',
  'min_samples_leaf': 8,
  'min_samples_split': 10,
  'n_estimators': 400,
  'subsample': 0.85},
 'AdaBoost': {'estimator__max_depth': 4,
  'estimator__min_samples_leaf': 4,
  'estimator__min_samples_split': 10,
  'learning_rate': 0.1,
  'loss': 'linear',
  'n_estimators': 600},
 'XGBoost': {'colsample_bytree': 0.85,
  'learning_rate': 0.02,
  'max_depth': 4,
  'min_child_weight': 2,
  'n_estimators': 1100,
  'reg_alpha': 0.1,
  'reg_lambda': 5.0,
  'subsample': 0.85}}